In [1]:
from pathlib import Path
import duckdb

ROOT = Path(r"D:\Driveguard")
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
TMP = ROOT / "data" / "tmp"
PROCESSED.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)

# SMART attributes we keep: bad sectors, age, errors, timeouts, temperature
SMART = [5, 9, 187, 188, 194, 197, 198]

In [2]:
for p in RAW.iterdir():
    print(p.name)

data_Q1_2026
data_Q2_2026
data_Q3_2025
data_Q4_2025


In [4]:
def ingest(quarter):
    raw = RAW / quarter
    out = PROCESSED / f"{quarter}.parquet"

    files = [p.as_posix() for p in raw.rglob("*.csv")
             if "__MACOSX" not in p.parts and not p.name.startswith("._")]
    if not files:
        print(f"No CSV files found in {raw}")
        return
    print(f"Found {len(files)} daily files in {quarter}")

    smart_cols = ",\n".join(
        f"TRY_CAST(smart_{n}_raw AS DOUBLE) AS smart_{n}_raw" for n in SMART
    )

    con = duckdb.connect()
    con.execute(f"SET temp_directory='{TMP.as_posix()}'")
    con.execute(f"""
    COPY (
      SELECT
        TRY_CAST("date" AS DATE)           AS "date",
        serial_number,
        model,
        TRY_CAST(capacity_bytes AS BIGINT) AS capacity_bytes,
        TRY_CAST(failure AS INTEGER)       AS failure,
        {smart_cols}
      FROM read_csv({files}, all_varchar=true, union_by_name=true, header=true)
    ) TO '{out.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    rows, drives, fails = con.execute(f"""
      SELECT COUNT(*), COUNT(DISTINCT serial_number), SUM(failure)
      FROM '{out.as_posix()}'
    """).fetchone()
    print(f"{quarter}: {rows:,} rows | {drives:,} drives | {fails:,} failures")
    print(f"Saved -> {out}")

In [5]:
  for q in ["data_Q2_2026", "data_Q1_2026", "data_Q4_2025", "data_Q3_2025"]:
      ingest(q)
      print("-" * 60)

Found 91 daily files in data_Q2_2026


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

data_Q2_2026: 31,968,003 rows | 359,101 drives | 1,522 failures
Saved -> D:\Driveguard\data\processed\data_Q2_2026.parquet
------------------------------------------------------------
Found 90 daily files in data_Q1_2026


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

data_Q4_2025: 30,941,708 rows | 341,664 drives | 945 failures
Saved -> D:\Driveguard\data\processed\data_Q4_2025.parquet
------------------------------------------------------------
Found 92 daily files in data_Q3_2025


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

data_Q3_2025: 29,844,451 rows | 332,915 drives | 1,254 failures
Saved -> D:\Driveguard\data\processed\data_Q3_2025.parquet
------------------------------------------------------------
